In [0]:
%sql
USE CATALOG kafka;
CREATE SCHEMA IF NOT EXISTS banking;
USE SCHEMA banking;

In [0]:
%sql
CREATE OR REPLACE TABLE users (
    user_id INT,
    name STRING,
    balance DOUBLE
)
USING DELTA;

INSERT INTO users VALUES
(1,'Hritik',1000),
(2,'Nikhil',500),
(3,'Tanmay',200);

In [0]:
%sql
SELECT * from users;

In [0]:
%sql
CREATE OR REPLACE TABLE transactions (
    txn_id STRING,
    sender INT,
    receiver INT,
    amount DOUBLE,
    txn_time TIMESTAMP
)
USING DELTA;

In [0]:
%sql
UPDATE users
SET balance = balance - 200
WHERE user_id = 1;

UPDATE users
SET balance = balance + 200
WHERE user_id = 2;

INSERT INTO transactions VALUES
('txn_001',1,2,200,current_timestamp());

In [0]:
%sql
CREATE OR REPLACE PROCEDURE transfer_money(sender_id INT, receiver_id INT, amount DOUBLE)
LANGUAGE SQL
SQL SECURITY INVOKER
BEGIN

  DECLARE sender_balance  DOUBLE;
  DECLARE err_message     STRING;

  -- ── STEP 1: Get sender's current balance ──
  SET sender_balance = (
    SELECT balance FROM users WHERE user_id = sender_id
  );

  -- ── STEP 2: Validate sender exists ──
  IF sender_balance IS NULL THEN
    SET err_message = CONCAT('Sender account not found: ', sender_id);
    SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = err_message;
  END IF;

  -- ── STEP 3: Validate receiver exists ──
  IF NOT EXISTS (SELECT 1 FROM users WHERE user_id = receiver_id) THEN
    SET err_message = CONCAT('Receiver account not found: ', receiver_id);
    SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = err_message;
  END IF;

  -- ── STEP 4: Validate amount ──
  IF amount <= 0 THEN
    SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Amount must be greater than zero';
  END IF;

  -- ── STEP 5: Check sufficient balance BEFORE any update ──
  IF sender_balance < amount THEN
    SET err_message = CONCAT('Insufficient balance. Available: ', sender_balance, ', Required: ', amount);
    SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = err_message;
  END IF;

  -- ── STEP 6: Safe to transfer now ──
  UPDATE users
  SET balance = balance - amount
  WHERE user_id = sender_id;

  UPDATE users
  SET balance = balance + amount
  WHERE user_id = receiver_id;

  INSERT INTO transactions VALUES (
    CONCAT('txn_', CAST(current_timestamp() AS STRING)),
    sender_id,
    receiver_id,
    amount,
    current_timestamp()
  );

END;

In [0]:
%sql
-- Enable catalogManaged feature on your tables
ALTER TABLE kafka.banking.users
SET TBLPROPERTIES ('delta.feature.catalogManaged' = 'supported');

ALTER TABLE kafka.banking.transactions
SET TBLPROPERTIES ('delta.feature.catalogManaged' = 'supported');

In [0]:
%sql
call transfer_money(1,2,900);
    
SELECT * FROM transactions;

In [0]:
%sql
SELECT * FROM users;